# EDA — Health Canada submissions under review

Short exploration behind `docs/methodology.md`. Run from the project root or
adjust `sys.path` below.

- distribution of `review_days` (overall and by class flag / submission type)
- outcome / approval balance
- categorical cardinality
- the single negative-delay row

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import CleaningReport, load_completed

report = CleaningReport()
df = load_completed(report)
report.as_dict()

## review_days distribution

In [ ]:
df['review_days'].describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df['review_days'].clip(upper=1000).hist(bins=50, ax=ax[0])
ax[0].set_title('review_days (clipped at 1000)')
df.groupby('submission_type')['review_days'].plot(kind='kde', ax=ax[1], legend=True)
ax[1].set_xlim(0, 1200)
ax[1].set_title('by submission type')
plt.tight_layout()

## Median review_days by submission-class flag

In [ ]:
rows = []
for flag in C.CLASS_FLAG_FEATURES + ['is_supplemental']:
    on = df.loc[df[flag] == 1, 'review_days']
    rows.append({'flag': flag, 'n': len(on),
                 'median_with': on.median(),
                 'median_without': df.loc[df[flag] == 0, 'review_days'].median()})
pd.DataFrame(rows).sort_values('median_with')

## Outcome and approval balance

In [ ]:
display(df['outcome'].value_counts())
display(df.groupby('submission_type')['approved'].agg(['mean', 'count']))

## Categorical cardinality

In [ ]:
df[['company_name', 'therapeutic_area']].nunique()

## The negative-delay row

Dropped by `load_completed`; shown here straight from the raw sheet.

In [ ]:
raw = pd.read_excel(C.RAW_FILE, sheet_name="Supplemental sub's completed", header=C.HEADER_ROW)
raw.loc[raw[C.COL_CONTROL_NUMBER] == 300215,
        [C.COL_CONTROL_NUMBER, C.COL_DATE_ACCEPTED, C.COL_DATE_CONCLUDED, C.COL_OUTCOME]]